# Project 2 · Module 4: question answering (KGQA) benchmark

120 template questions (12 types × 10) with gold answers computed from the KG, a Gemini paraphrase of each, and two planners scored on both forms:
- **template planner**: regular expressions, no LLM (baseline);
- **LLM planner**: an open-weight model on Groq (default `openai/gpt-oss-120b`) turns the question into a JSON query plan.

In both cases the plan runs on the graph and the answer comes from the graph only. Before running, provide two API keys, as Colab secrets (key icon in the left bar) or as environment variables with the same names: `GeminiAPIKey` (Google AI Studio) and `GroqAPIKey` (console.groq.com, free, no card). Gemini's free Flash tier allows only about 20 requests a day, too few for 240 planner calls; Groq's free tier allows about 1,000 requests and 200K tokens a day per model. Every LLM reply is cached in `data/interim/llm_cache/`, so a rerun or a resumed run costs nothing.

If the free quota runs out, the cell stops with `LLMUnavailable`. Wait for the quota to reset and run all cells again: every reply already received is read from the cache, so it continues where it stopped.

In [ ]:
# Setup: run from a clone of the repo, locally or in Colab
import os, sys, json
from pathlib import Path

REPO_URL = "https://github.com/<your-account>/<repo>.git"   # Colab only: the repo to clone
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("/content/repo").exists():
    !git clone -q {REPO_URL} /content/repo
ROOT = Path("/content/repo") if IN_COLAB else Path.cwd().resolve().parents[1]   # notebooks/ -> extension -> repo
EXT = ROOT / "equity-research-extension"   # this extension: code, reference data, outputs

# The Phase 1 KG (read-only): the folder holding fused_edges.jsonl and canonical_entities.jsonl
KG = next((p for p in (ROOT / "data/kg", ROOT / "semiconductor_kg/data/kg") if (p / "fused_edges.jsonl").exists()), Path())
KG = Path(os.environ.get("P1_KG_DIR", KG))
assert (EXT / "src/kg/load.py").exists(), f"extension code not found under {EXT}"
assert (KG / "fused_edges.jsonl").exists(), "Phase 1 KG not found: set P1_KG_DIR to the folder with fused_edges.jsonl"
os.environ["P1_KG_DIR"] = str(KG)
if str(EXT) not in sys.path:
    sys.path.insert(0, str(EXT))

In [ ]:
!pip -q install google-genai openai

In [ ]:
import importlib; importlib.invalidate_caches()
import pandas as pd
from src.kg import load, cluster, qa, qa_bench, llm
for m in (cluster, qa, qa_bench, llm):
    importlib.reload(m)
kg = load.load()
G = qa.Graph(kg)
OUT = EXT / 'data/results/qa'   # frozen benchmarks and saved results

# ---- settings -----------------------------------------------------------------------------------------------
SEED = 11            # 7 = development benchmark (prompt rules written from its errors); 11 = held-out test benchmark
RUNS = [('llm', 2, 'paraphrase')]   # (planner, prompt version, question form); prompt 2 is the final planner prompt
# Groq's free tier is about 200K tokens a day, roughly 240 planner calls. Replies are cached, so a rerun is free.
# -------------------------------------------------------------------------------------------------------------


RUN = OUT / f'seed{SEED}'
RUN.mkdir(parents=True, exist_ok=True)
if (RUN / 'benchmark.jsonl').exists():   # a benchmark is frozen once written: ids, paraphrases and gold stay fixed
    bench = qa_bench.load_bench(RUN / 'benchmark.jsonl')
else:
    dev = OUT / 'seed7' / 'benchmark.jsonl'   # a held-out benchmark shares no question with the development one
    avoid = qa_bench.load_bench(dev) if SEED != 7 and dev.exists() else None
    bench = qa_bench.build(kg, RUN / 'benchmark.jsonl', seed=SEED, avoid=avoid)
print(f'seed {SEED}: {len(bench)} questions;', sum(e['surface'] != kg.entities[e['id']]['canonical_name'] for q in bench for e in q['entities']), 'entity mentions use an alias')
display(pd.DataFrame(bench)[['id', 'type', 'question', 'gold_names']].groupby('type').head(1))

## Paraphrases (Gemini Flash-Lite)
A model from a different family than the planner writes the paraphrases, so the test questions are not in the planner's own style. A paraphrase is kept only if every company or product name survives verbatim; otherwise the template question stays (flagged).

In [ ]:
def secret(name):   # Colab secret if available, else an environment variable of the same name
    try:
        from google.colab import userdata
        return userdata.get(name)
    except ImportError:
        return os.environ[name]

GEMINI_KEY = secret('GeminiAPIKey')
PARAPHRASE_MODEL = 'gemini-3.5-flash-lite'   # cheap, high rate limit
PLANNER_MODEL = 'openai/gpt-oss-120b'        # on Groq; alternatives: 'openai/gpt-oss-20b', 'qwen/qwen3-32b'
cache = EXT / 'data/interim/llm_cache'
para_llm = llm.Gemini(GEMINI_KEY, PARAPHRASE_MODEL, cache, pause=4.0, temperature=0.7)

todo = [q for q in bench if not q.get('paraphrase')]   # only questions without one (a frozen benchmark keeps its paraphrases)
for i in range(0, len(todo), 20):
    batch = todo[i:i + 20]
    try:
        got = qa.parse_plan(para_llm(qa_bench.paraphrase_prompt(batch)))
    except (ValueError, qa.PlanError) as e:   # unreadable reply; an API/quota error stops the cell instead
        print('batch', i, 'unreadable reply:', e); got = {}
    for q in batch:
        p = str(got.get(q['id'], ''))
        q['paraphrase_ok'] = qa_bench.check_paraphrase(q, p)
        q['paraphrase'] = p if q['paraphrase_ok'] else q['question']
for q in [q for q in todo if not q['paraphrase_ok']]:   # one retry, question by question
    try:
        p = str(qa.parse_plan(para_llm(qa_bench.paraphrase_prompt([q]) + '\n(retry)')).get(q['id'], ''))
    except (ValueError, qa.PlanError):
        p = ''
    if qa_bench.check_paraphrase(q, p):
        q['paraphrase_ok'], q['paraphrase'] = True, p
(RUN / 'benchmark.jsonl').write_text('\n'.join(json.dumps(q, ensure_ascii=False) for q in bench) + '\n')
print(sum(q['paraphrase_ok'] for q in bench), 'of', len(bench), 'paraphrases kept')
display(pd.DataFrame(bench)[['id', 'question', 'paraphrase', 'paraphrase_ok']].sample(8, random_state=1))

## Evaluate
The planner prompt's rules were written from the errors on the seed-7 development benchmark, so the reported accuracy comes from seed 11, a held-out benchmark generated without overlap with seed 7. Seed 7's report can be rebuilt any time without API calls: set `SEED = 7`, `RUNS = []`. Benchmarks and results are saved in `data/results/qa/`.

In [ ]:
plan_llm = llm.Groq(secret('GroqAPIKey'), PLANNER_MODEL, cache)   # 8 s pause: stays under 8K tokens/min
planners = {'template': lambda v: qa_bench.template_planner,
            'llm': lambda v: (lambda question: qa.parse_plan(plan_llm(qa.planner_prompt(question, v))))}
save = lambda name, rows: (RUN / f"results_{name.replace('/', '_')}.jsonl").write_text(
    '\n'.join(json.dumps(r, ensure_ascii=False, default=str) for r in rows) + '\n')

for field in ('question', 'paraphrase'):                       # the template baseline is free: always run it
    save(f'template/{field}', qa_bench.run(bench, qa_bench.template_planner, G, field))
for kind, version, field in RUNS:
    name = f'{kind}-v{version}/{field}'
    save(name, qa_bench.run(bench, planners[kind](version), G, field))
    print(f'{name}: done ({plan_llm.calls} new LLM calls so far)')

# report over every run saved for this benchmark, scored again with the current scoring
runs = {}
for f in sorted(RUN.glob('results_*.jsonl')):
    name = f.stem[len('results_'):].replace('_', '/', 1)
    runs[name] = qa_bench.rescore([json.loads(l) for l in f.read_text().splitlines() if l.strip()], bench)
EX = qa_bench.KNOWN_DEFECTS.get(SEED, {})
report = {'seed': SEED, 'excluded': EX, 'models': {'paraphrase': PARAPHRASE_MODEL, 'planner': PLANNER_MODEL}}
report.update({k: qa_bench.summary(v, EX) for k, v in runs.items()})
pairs = [('template/paraphrase', k) for k in runs if k.startswith('llm') and k.endswith('/paraphrase')]
pairs += [(k.replace('/paraphrase', '/question'), k) for k in runs if k.endswith('/paraphrase') and k.replace('/paraphrase', '/question') in runs and k.startswith('llm')]
report['mcnemar'] = {f'{a} vs {b}': qa_bench.mcnemar(runs[a], runs[b], EX) for a, b in pairs}
(RUN / 'qa_report.json').write_text(json.dumps(report, indent=1))

display(pd.DataFrame({k: {m: report[k][m] for m in ('n', 'accuracy', 'ci95', 'mean_f1', 'errors')} for k in runs}).T)
display(pd.DataFrame({k: report[k]['by_type'] for k in runs}))
print(json.dumps(report['mcnemar'], indent=1))

In [ ]:
# Where the LLM planner goes wrong on paraphrases 
worst = sorted(k for k in runs if k.startswith('llm') and k.endswith('/paraphrase'))[-1]
bad = [r for r in runs[worst] if not r['correct'] and r['id'] not in EX]
gold = {q['id']: q for q in bench}
print(worst, len(bad), 'wrong')
display(pd.DataFrame([{'id': r['id'], 'type': r['type'], 'question': r['question'], 'error': r.get('error'),
                       'f1': r['f1'], 'plan': json.dumps(r['plan'])[:160], 'gold plan': json.dumps(gold[r['id']]['plan'])[:160]} for r in bad]))

Reading the results: the template planner is 100% on template questions by construction (sanity check only). The comparison that matters is paired (same questions): LLM vs template planner on paraphrases, tested with McNemar on the questions where the two disagree. Set answers ignore the entities named in the question (a company can sit in its own supply loop). With about 120 questions the 95% interval on an accuracy is roughly ±6–8 points.